In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text,remove_diacritics

In [2]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
unmatched_file = Path(project_root / "data/people/unmatched_flat.json")
nopes_manual_file = Path(project_root / "scripts/notebooks/nopes_manual_edits.json")
singles_checked_file = Path(project_root / "data/people/messy_shit/inbetweens/check_singles.json")
merged_file = Path(project_root / "data/people/matches_merged.json")


In [3]:
with open(nopes_manual_file, "r") as f:
   nopes_editing = json.load(f)
# rprint(dict(list(nopes_editing.items())[20:500:30]))
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)

with open(singles_checked_file, "r") as f:
   singles_checked = json.load(f)
with open(merged_file, "r") as f:
   merged = json.load(f)


In [4]:

# check_names = {}
# rprint(f"names: {len(nopes_editing)}")

# for name, entry in nopes_editing.items():
#     last = entry["family_name"]
#     first = entry["given_names"]
#     single = entry["single_name"]
#     name_string = (last or "") + (first or "") + (single or "")
#     # rprint(name_string)

#     for char in name_string:
#         if not char.isalpha():
#             if char.isspace():
#                 continue

#             check_names[name] = entry

# rprint(f"names with issues: {len(check_names)}")

# with open("check_names.json", "w") as f:
#     json.dump(check_names, f, ensure_ascii=False, indent=2)


In [5]:
unmatched_dict = {}
# rprint(f"unmatched entries: {len(unmatched)}")
# total_unmatched_flat = len(unmatched)
# total_nopes = len(nopes)

for entry in unmatched:
    composite_id = entry["composite_id"]

    if composite_id not in unmatched_dict:
        unmatched_dict[composite_id] = [entry]
    else:
        unmatched_dict[composite_id].append(entry)
rprint(dict(list(unmatched_dict.items())[:1]))


{
    'wien_210_9_10': [
        {
            'composite_id': 'wien_210_9_10',
            'display_name': 'ROMMEL, Otto',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False,
            'is_organisation': False,
            'display_norm': 'rommel, otto',
            'last': 'ROMMEL',
            'first': 'Otto',
            'single': None,
            'original_entry': 'ROMMEL, Otto\nEin Jahrhundert Alt-Wiener Parodie. Herausgegeben von … Wien 
Österreichischer Bundesverlag 1930. 285 S. OLn.',
            'verification_notes': "The entry states 'Herausgegeben von …' but no editor name follows the ellipsis. 
missing_person flag set accordingly. Note: ROMMEL, Otto is listed as the main entry (author/compiler), but the 
'Herausgegeben von …' suggests an editor role may be missing or that Rommel himself is the editor with the name 
omitted by the cataloger.",
            'match_found': False,
            'person_id': None
        }
    ]
}

In [6]:
# rprint(dict(list(merged.items())[:1]))


In [7]:
# Dict comprehension to shape my data and separate it into separate data structures

singles_found = {k: v["info"] for k, v in singles_checked.items() if v["info"]["person_id"] is not None}
# rprint(f"existing singles: {len(singles_found)}")

# new_singles = {k: v["info"] for k, v in singles_checked.items() if v["info"]["person_id"] is None}
# rprint(f"new_singles count: {len(new_singles)}")

# rprint(f"new_singles: {new_singles}")
# rprint(dict(list(singles_found.items())[:1]))

# matches2merge = {}
id_in_merged = 0
not_in_merged = 0

for name, entry in singles_found.items():
    person_id = str(entry["person_id"])
    display_norm = name
    for composite_id in entry["composite_id"]:
        records = unmatched_dict[composite_id]

    for u in records:
        if u["display_norm"] == name:
            display_name = u["display_name"]
            sort_order = u["sort_order"]
            is_author = u["is_author"]
            is_editor = u["is_editor"]
            is_contributor = u["is_contributor"]
            is_translator = u["is_translator"]

            if person_id not in merged:
                not_in_merged += 1
                merged[person_id] = [{
                    "display_name": display_name,
                    "composite_id": composite_id,
                    "sort_order": sort_order,
                    "is_author": is_author,
                    "is_editor": is_editor,
                    "is_contributor": is_contributor,
                    "is_translator": is_translator
                }]
            else:
                id_in_merged += 1
                existing = [e["composite_id"] for e in merged[person_id]]
                if composite_id not in existing:
                    merged[person_id].append({
                        "display_name": display_name,
                        "composite_id": composite_id,
                        "sort_order": sort_order,
                        "is_author": is_author,
                        "is_editor": is_editor,
                        "is_contributor": is_contributor,
                        "is_translator": is_translator,
                })
                break
# rprint(f"new id: {not_in_merged}")
# rprint(f"old id: {id_in_merged}")

# with open("matches_merged_with_singles.json", "w") as f:
#     json.dump(merged, f, ensure_ascii=False, indent=2)


In [ ]:
new_singles = {k: v["info"] for k, v in singles_checked.items() if v["info"]["person_id"] is None}
# rprint(f"new_singles count: {len(new_singles)}")

missing_names = {}

for name, entry in new_singles.items():
    # rprint(f"name, entry: {name}, {entry}")
    nopes = nopes_editing[name]
    # rprint(f"nopes display name: {nopes["display_name"]}")
    # rprint(f"entry display name: {entry["display_name"]}")

    nopes["last_norm"] = entry["last_norm"]
    nopes["family_name"] = entry["family_name"]
    nopes["first_norm"] = entry["first_norm"]
    nopes["given_names"] = entry["given_names"]
    nopes["single_norm"] = entry["single_norm"]
    nopes["single_name"] = entry["single_name"]
    nopes["notes"] = entry["notes"]

    if entry.get("name_particles"):
        nopes["name_particles"] = entry["name_particles"]
    if entry.get("is_organisation"):
        nopes["is_organisation"] = entry["is_organisation"]

    if name not in nopes_editing:
        rprint(f"missing name found!")
        missing_names[name] = entry

    # nopes[name] = entry

# with open("nopes_with_singles.json", "w") as f:
#     json.dump(nopes_editing, f, ensure_ascii=False, indent=2)
# with open("missing_names.json", "w") as f:
#     json.dump(missing_names, f, ensure_ascii=False, indent=2)
